In [3]:
import fitz
import pandas as pd
import re
import os

files = {
    "Овощи": "./Цены на овощи и фрукты/овощи.pdf",
    "Фрукты": "./Цены на овощи и фрукты/фрукты.pdf"
}

pdf_data = {}

for name, path in files.items():
    doc = fitz.open(path)

    pages = []
    for page in doc:
        pages.append(page.get_text())

    pdf_data[name] = {
        "pages": pages,
        "text": "\n".join(pages),
        "page_count": len(doc)
    }

    print(f"{name}:")
    print(f"Количество страниц: {len(doc)}")
    print(f"Количество символов: {len(pdf_data[name]['text'])}")
    print(f"Количество строк: {len(pdf_data[name]['text'].splitlines())}")
    print()

Овощи:
Количество страниц: 7
Количество символов: 27495
Количество строк: 344

Фрукты:
Количество страниц: 6
Количество символов: 22228
Количество строк: 269



In [4]:
for name, data in pdf_data.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)

    print(data["text"][:2500])


Овощи
New York Terminal Market Vegetables Prices (NX_FV020)
November 06, 2026
Agricultural Marketing Service
Specialty Crops Market News
Page 1
NX_FV020
Email us with accessibility issues with this report.
HERBS
---ANISE:  MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24.00-26.00 24s 22.00-24.00 fine appearance 26.00-30.00 mostly 28.00 
---ARUGULA:  MARKET STEADY. cartons bunched NEW JERSEY 24s 22.00-24.00 with root 4/5 bushel crates bunched FLORIDA 24s 24.00 
with root cartons 2 1-1/2 lb film bags FLORIDA BABY TYPE 16.00-17.00 
---BASIL:  OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA air 12.00-14.00 bushel cartons bunched FLORIDA WITH ROOT 15s 36.00-
42.00 mostly 38.00-40.00 
---BAY LEAVES:  OFFERINGS LIGHT. repacked enroute 22.00-23.00 1 KG cartons film bag loose 1 lb film bags loose repacked enroute 
14.00 
---CELERIAC (CELERY ROOT):  MARKET ABOUT STEADY. 25 lb cartons MICHIGAN 20.00 20 lb cartons CANADA QUEBEC 15.00-16.00 
---CHIVES:  MARKET STEADY. 1 kg cartons bunched COLOMBIA gr

In [5]:
for name, data in pdf_data.items():
    lines = data["text"].splitlines()

    product_lines = [
        line.strip()
        for line in lines
        if line.strip().startswith("---")
    ]

    product_names = []

    for line in product_lines:
        match = re.match(r"---([^:]+):", line)

        if match:
            product_names.append(match.group(1).strip())

    print(f"\n{name}")
    print(f"Количество товарных блоков: {len(product_lines)}")
    print(f"Количество уникальных названий: {len(set(product_names))}")
    print(f"Количество повторяющихся названий: "
          f"{len(product_names) - len(set(product_names))}")


Овощи
Количество товарных блоков: 125
Количество уникальных названий: 124
Количество повторяющихся названий: 1

Фрукты
Количество товарных блоков: 57
Количество уникальных названий: 53
Количество повторяющихся названий: 4


In [6]:
price_pattern = re.compile(
    r"(?<!\d)\d+\.\d{2}(?:\s*-\s*\d+\.\d{2})?"
)

for name, data in pdf_data.items():
    lines = data["text"].splitlines()

    lines_with_prices = [
        line.strip()
        for line in lines
        if price_pattern.search(line)
    ]

    print(f"\n{name}")
    print(f"Строк с потенциальными ценовыми значениями: "
          f"{len(lines_with_prices)}")

    print("\nПримеры:")
    for line in lines_with_prices[:10]:
        print(line)


Овощи
Строк с потенциальными ценовыми значениями: 250

Примеры:
---ANISE:  MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24.00-26.00 24s 22.00-24.00 fine appearance 26.00-30.00 mostly 28.00
---ARUGULA:  MARKET STEADY. cartons bunched NEW JERSEY 24s 22.00-24.00 with root 4/5 bushel crates bunched FLORIDA 24s 24.00
with root cartons 2 1-1/2 lb film bags FLORIDA BABY TYPE 16.00-17.00
---BASIL:  OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA air 12.00-14.00 bushel cartons bunched FLORIDA WITH ROOT 15s 36.00-
42.00 mostly 38.00-40.00
---BAY LEAVES:  OFFERINGS LIGHT. repacked enroute 22.00-23.00 1 KG cartons film bag loose 1 lb film bags loose repacked enroute
14.00
---CELERIAC (CELERY ROOT):  MARKET ABOUT STEADY. 25 lb cartons MICHIGAN 20.00 20 lb cartons CANADA QUEBEC 15.00-16.00
---CHIVES:  MARKET STEADY. 1 kg cartons bunched COLOMBIA greenhouse 12s 17.00-18.00 1 lb film bags loose NONE repacked local
greenhouse 10.00-11.00

Фрукты
Строк с потенциальными ценовыми значениями: 181

Примеры:


In [7]:
import re
import pandas as pd

price_pattern = re.compile(r"(?<!\d)\d+\.\d{2}(?:\s*-\s*\d+\.\d{2})?")

for name, data in pdf_data.items():
    text = data["text"]

    # Разделяем текст на товарные блоки
    blocks = re.split(r"(?=^---)", text, flags=re.MULTILINE)
    blocks = [b.strip() for b in blocks if b.strip().startswith("---")]

    results = []

    for block in blocks:
        title_match = re.match(r"---([^:]+):", block)
        product = title_match.group(1).strip() if title_match else "Не определено"

        results.append({
            "Продукт": product,
            "Есть числовые значения": bool(price_pattern.search(block)),
            "Есть NONE": "NONE" in block.upper(),
            "Текст блока": block
        })

    result_df = pd.DataFrame(results)

    print(f"\n{name}")
    print("Всего товарных блоков:", len(result_df))
    print("Блоков с числовыми значениями:",
          result_df["Есть числовые значения"].sum())
    print("Блоков без числовых значений:",
          (~result_df["Есть числовые значения"]).sum())
    print("Блоков с обозначением NONE:",
          result_df["Есть NONE"].sum())

    print("\nБлоки без числовых значений:")
    display(
        result_df.loc[
            ~result_df["Есть числовые значения"],
            ["Продукт", "Текст блока"]
        ]
    )


Овощи
Всего товарных блоков: 125
Блоков с числовыми значениями: 125
Блоков без числовых значений: 0
Блоков с обозначением NONE: 24

Блоки без числовых значений:


,Продукт,Текст блока



Фрукты
Всего товарных блоков: 57
Блоков с числовыми значениями: 57
Блоков без числовых значений: 0
Блоков с обозначением NONE: 1

Блоки без числовых значений:


,Продукт,Текст блока


In [8]:
units = [
    "kg", "lb", "oz", "bushel", "carton",
    "crate", "bag", "sack", "flat", "box",
    "container", "jar"
]

for name, data in pdf_data.items():
    text = data["text"].lower()

    print(f"\n{name}")
    for unit in units:
        count = len(re.findall(rf"\b{unit}\b", text))
        if count > 0:
            print(f"{unit}: {count}")


Овощи
kg: 26
lb: 88
oz: 13
bushel: 57
carton: 1
bag: 6
flat: 3
container: 3
jar: 6

Фрукты
kg: 10
lb: 49
oz: 10
bushel: 12
carton: 1
flat: 1
container: 3


In [9]:
for name, data in pdf_data.items():
    blocks = re.split(
        r"(?=^---)", data["text"], flags=re.MULTILINE
    )
    blocks = [b.strip() for b in blocks if b.strip().startswith("---")]

    products = []

    for block in blocks:
        match = re.match(r"---([^:]+):", block)
        if match:
            products.append(match.group(1).strip())

    duplicates = pd.Series(products)
    duplicate_names = duplicates[duplicates.duplicated(keep=False)].unique()

    print(f"\n{name}")
    print("Повторяющиеся наименования:")

    for product in duplicate_names:
        print(f"\n{product}")
        for block in blocks:
            if block.startswith(f"---{product}:"):
                print(block)


Овощи
Повторяющиеся наименования:

GINGER ROOT
---GINGER ROOT:  MARKET STEADY. 30 lb cartons BRAZIL 32.00-33.00 fine appearance 34.00-36.00 CHINA 24.00-28.00 mostly 26.00 fine 
appearance 30.00-34.00 mostly 32.00 5 lb cartons NONE repacked local 8.00 10 lb cartons NONE repacked local 16.00
---GINGER ROOT:  MARKET STEADY. 30 lb cartons PERU 26.00-32.00 mostly 28.00-30.00 fair appearance holdovers 22.00-24.00 fine 
appearance 34.00-38.00 mostly 36.00 
USDA, AMS, Specialty Crops Market News
Bronx, NY 10474
Phone (718) 542-2225 | https://mymarketnews.ams.usda.gov/
Source:

Фрукты
Повторяющиеся наименования:

BLACKBERRIES
---BLACKBERRIES:  MARKET SLIGHTLY LOWER. flats 12 6-oz cups with lids CALIFORNIA large fine appearance 34.00 MEXICO large 20.00-
22.00
---BLACKBERRIES:  MARKET SLIGHTLY LOWER. flats 12 6-oz cups with lids CALIFORNIA large 28.00-30.00 fair appearance holdovers 12.00-
16.00 mostly 14.00 MEXICO large fair appearance holdovers 18.00

RASPBERRIES
---RASPBERRIES:  MARKET SLIGHT